In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))


import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision.transforms.functional as TF
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


from src.dataset.bbbc038 import BBBC038Dataset, SegmentationTransform
from src.nn.models import ConvBlock, train_loop, test_loop
from src.nn.optimizers import create_optimizer
from src.plot.plot import plot_resultados
from src.nn.metrics import *
from src.nn.loss import l1_loss, l2_loss
from src.nn.targets import instance_map_to_targets

In [ ]:
class UNetModified(nn.Module):
    """
    U-Net com duas cabeças:

    1. center_head:
       prediz um heatmap de centros [N, 1, H, W]

    2. offset_head:
       prediz offsets (dx, dy) [N, 2, H, W]
    """

    def __init__(self):
        super().__init__()

        # =====================================================
        # ENCODER
        # =====================================================

        self.enc1 = ConvBlock(3, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)

        self.pool = nn.MaxPool2d(2, 2)

        # =====================================================
        # DECODER
        # =====================================================

        self.up3 = nn.ConvTranspose2d(
            256, 128, 2, 2
        )

        self.dec3 = ConvBlock(
            384, 128
        )

        self.up2 = nn.ConvTranspose2d(
            128, 64, 2, 2
        )

        self.dec2 = ConvBlock(
            192, 64
        )

        self.up1 = nn.ConvTranspose2d(
            64, 32, 2, 2
        )

        self.dec1 = ConvBlock(
            96, 32
        )

        # =====================================================
        # HEADS
        # =====================================================

        # Um valor por pixel
        self.center_head = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

        # dx, dy por pixel
        self.offset_head = nn.Conv2d(
            32,
            2,
            kernel_size=1
        )

    def forward(self, x):

        # =====================================================
        # ENCODER
        # =====================================================

        e1 = self.enc1(x)
        p1 = self.pool(e1)

        e2 = self.enc2(p1)
        p2 = self.pool(e2)

        e3 = self.enc3(p2)
        p3 = self.pool(e3)

        # =====================================================
        # DECODER
        # =====================================================

        d3 = self.up3(p3)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # =====================================================
        # OUTPUTS
        # =====================================================

        center = self.center_head(d1)

        offsets = self.offset_head(d1)

        return center, offsets

In [ ]:
def train_center_offset(
    dataloader,
    device,
    model,
    optimizer
):

    model.train()

    total_loss = 0.0

    for X, instance_map in dataloader:

        X = X.to(device)
        instance_map = instance_map.to(device)

        # =============================================
        # TARGETS
        # =============================================

        center_target = []
        offset_target = []
        foreground_target = []

        for i in range(
            instance_map.shape[0]
        ):

            center, offset, foreground = (
                instance_map_to_targets(
                    instance_map[i]
                    .cpu()
                    .numpy()
                )
            )

            center_target.append(center)
            offset_target.append(offset)
            foreground_target.append(foreground)

        center_target = torch.stack(
            center_target
        ).to(device)

        offset_target = torch.stack(
            offset_target
        ).to(device)

        foreground_target = torch.stack(
            foreground_target
        ).to(device)

        # =============================================
        # FORWARD
        # =============================================

        center_pred, offset_pred = model(X)

        # =============================================
        # LOSSES
        # =============================================

        center_loss = F.mse_loss(
            center_pred.sigmoid(),
            center_target
        )

        # Offset só deve ser penalizado em foreground
        foreground_mask = (
            foreground_target
            .unsqueeze(1)
            .expand_as(offset_pred)
        )

        offset_loss = F.l1_loss(
            offset_pred[
                foreground_mask
            ],
            offset_target[
                foreground_mask
            ]
        )

        # =============================================
        # LOSS TOTAL
        # =============================================

        loss = (
            center_loss
            +
            offset_loss
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
center_loss = l2_loss(
    center_pred.sigmoid(),
    center_target
)